# JustDoWork connection check
Run both cells from the project folder. The notebook reads `.env` (or existing environment variables), never displays your key, and makes one small paid API request.

In [ ]:
import os
from pathlib import Path
import httpx

def setting(name):
    if os.environ.get(name):
        return os.environ[name]
    env_file = Path('.env')
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name:
                return value.strip().strip(chr(34)).strip(chr(39))
    return ''

api_key = setting('JUSTDOWORK_API_KEY')
base_url = (setting('JUSTDOWORK_BASE_URL') or 'https://api.justwoker.icu').rstrip('/')
model = 'claude-opus-4-8'  # Change only if your JustDoWork account lists another model.
endpoint = base_url + ('' if base_url.endswith('/v1') else '/v1') + '/messages'
print('Key configured:', bool(api_key))
print('Endpoint:', endpoint)
print('Model:', model)


In [ ]:
if not api_key:
    print('FAIL: Set JUSTDOWORK_API_KEY in .env or your environment.')
else:
    try:
        response = httpx.post(
            endpoint,
            headers={'x-api-key': api_key, 'anthropic-version': '2023-06-01'},
            json={'model': model, 'max_tokens': 16,
                  'messages': [{'role': 'user', 'content': 'Reply OK only.'}]},
            timeout=30,
        )
        print('HTTP status:', response.status_code)
        if 'application/json' not in response.headers.get('content-type', ''):
            print('FAIL: Endpoint returned non-JSON content; check the base URL.')
        else:
            body = response.json()
            if response.is_success and body.get('content'):
                reply = ' '.join(block.get('text', '') for block in body['content']
                                 if block.get('type') == 'text')
                print('PASS: Model replied:', reply[:100])
            else:
                error = body.get('error', {})
                print('FAIL:', str(error.get('message', error))[:300])
    except httpx.HTTPError as exc:
        print('FAIL: Network request failed:', type(exc).__name__)
